**STRUCTURED OUTPUTS**

**NORMAL TEXT OUTPUT - This is the default. Read the answer with result.raw.**

In [1]:
from crewai import Agent, Task, Crew, LLM

agent = Agent(
    role="AI Teacher",
    goal="Explain AI concepts simply",
    backstory="You teach beginners.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Explain what an AI agent is in two sentences.",
    expected_output="Two simple sentences.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
result = await crew.kickoff_async()

print(result.raw)

An AI agent is a computer program that can perceive its environment and take actions to achieve specific goals. It uses algorithms and data to make decisions, learn from experiences, and improve its performance over time.


**-result.raw → a string (str)**<br>
**-Best when you just want to display an answer.**

**JSON OUTPUT - output_json takes a Pydantic model that defines the JSON fields. Access the resulting values through result.json_dict**

In [2]:
from crewai import Agent, Task, Crew, LLM
from pydantic import BaseModel

class AIAnswer(BaseModel):
    topic: str
    explanation: str

agent = Agent(
    role="AI Teacher",
    goal="Explain AI concepts simply",
    backstory="You teach beginners.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Explain what an AI agent is.",
    expected_output="A topic and a short explanation.",
    agent=agent,
    output_json=AIAnswer
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
result = await crew.kickoff_async()

print(result.json_dict)
print(result.json_dict["explanation"])

{'topic': 'AI Agent', 'explanation': 'An AI agent is a computer program that can make decisions and take actions based on its environment. It uses algorithms and data to learn from experiences and improve its performance over time. AI agents can be simple, like a chatbot that answers questions, or complex, like self-driving cars that navigate roads.'}
An AI agent is a computer program that can make decisions and take actions based on its environment. It uses algorithms and data to learn from experiences and improve its performance over time. AI agents can be simple, like a chatbot that answers questions, or complex, like self-driving cars that navigate roads.


**-result.json_dict → a dictionary (dict)**<br>
**-Access a field with result.json_dict["explanation"].**<br>
**-Best when another part of your program needs named fields.**

**PYDANTIC OUTPUT - output_pydantic returns a model object. You can access its fields with dot notation.**

In [3]:
from crewai import Agent, Task, Crew, LLM
from pydantic import BaseModel

class AIAnswer(BaseModel):
    topic: str
    explanation: str

agent = Agent(
    role="AI Teacher",
    goal="Explain AI concepts simply",
    backstory="You teach beginners.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Explain what an AI agent is.",
    expected_output="A topic and a short explanation.",
    agent=agent,
    output_pydantic=AIAnswer
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
result = await crew.kickoff_async()

print(result.pydantic)
print(result.pydantic.explanation)

topic='AI Agent' explanation='An AI agent is a computer program that can make decisions and take actions based on its environment. It uses algorithms and data to learn from experiences and improve its performance over time. AI agents can be simple, like a chatbot that answers questions, or complex, like self-driving cars that navigate traffic.'
An AI agent is a computer program that can make decisions and take actions based on its environment. It uses algorithms and data to learn from experiences and improve its performance over time. AI agents can be simple, like a chatbot that answers questions, or complex, like self-driving cars that navigate traffic.


**-result.pydantic → an AIAnswer model object**<br>
**-Access a field with result.pydantic.explanation.**<br>
**-Unlike the JSON example’s dictionary, the result is an instance of the model you defined.**

**OUTPUT FILES - Set output_file on the task to save its answer when the crew runs. This example creates ai_agent_notes.md in the notebook’s current working folder**

In [4]:
from crewai import Agent, Task, Crew, LLM

agent = Agent(
    role="AI Teacher",
    goal="Write simple study notes",
    backstory="You prepare notes for beginners.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Write three short study notes about AI agents.",
    expected_output="Three Markdown bullet points.",
    agent=agent,
    output_file="ai_agent_notes.md"
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
await crew.kickoff_async()

print("Saved to ai_agent_notes.md")

Saved to ai_agent_notes.md


**-output_file="ai_agent_notes.md" → saves the task answer to a file.**<br>
**-This controls where the answer is stored. You can still access the answer in Python through the task or crew output.**

**ACCESSING TASK OUTPUTS - After execution, task.output holds that task’s TaskOutput. For a crew with several tasks, result.tasks_output holds their outputs in order.**

In [5]:
from crewai import Agent, Task, Crew, Process, LLM

agent = Agent(
    role="AI Teacher",
    goal="Teach AI concepts step by step",
    backstory="You explain concepts to beginners.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task1 = Task(
    description="Define an AI agent in one sentence.",
    expected_output="One simple sentence.",
    agent=agent
)

task2 = Task(
    description="Give one everyday example of an AI agent.",
    expected_output="One short example.",
    agent=agent,
    context=[task1]
)

crew = Crew(
    agents=[agent],
    tasks=[task1, task2],
    process=Process.sequential,
    verbose=False
)

result = await crew.kickoff_async()

print("First task:", task1.output.raw)
print("Second task:", task2.output.raw)

for number, task_output in enumerate(result.tasks_output, start=1):
    print(f"Task {number}: {task_output.raw}")

First task: An AI agent is a computer program that can perceive its environment, make decisions, and take actions to achieve specific goals.
Second task: One everyday example of an AI agent is a virtual personal assistant, like Siri or Google Assistant. These AI agents can perceive voice commands, understand user requests, make decisions about how to respond, and take actions such as setting reminders, playing music, or providing weather updates to help users achieve their goals.
Task 1: An AI agent is a computer program that can perceive its environment, make decisions, and take actions to achieve specific goals.
Task 2: One everyday example of an AI agent is a virtual personal assistant, like Siri or Google Assistant. These AI agents can perceive voice commands, understand user requests, make decisions about how to respond, and take actions such as setting reminders, playing music, or providing weather updates to help users achieve their goals.


**-task1.output.raw → answer from task 1**<br>
**-task2.output.raw → answer from task 2**<br>
**-result.tasks_output → a list of all task outputs, in task order**<br>
**The loop repeats the same two answers to show another way to access them.**

**ACCESSING THE FINAL CREW OUTPUT - kickoff_async() returns a CrewOutput. Its raw value is the final answer, based on the last task. If that last task uses output_json or output_pydantic**

In [6]:
from crewai import Agent, Task, Crew, LLM

agent = Agent(
    role="AI Teacher",
    goal="Give clear final answers",
    backstory="You explain AI to beginners.",
    llm=LLM(model="gpt-4o-mini", temperature=0),
    verbose=False
)

task = Task(
    description="Explain one benefit of using multiple AI agents.",
    expected_output="One beginner-friendly sentence.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)
result = await crew.kickoff_async()

print("Final answer:", result.raw)

Final answer: One benefit of using multiple AI agents is that they can specialize in different tasks, allowing for more efficient problem-solving and improved overall performance.


**-result.raw → the final crew answer**<br>
**-In a crew with multiple tasks, this normally reflects the last task’s output.**